# Environment

In [ ]:
import os
os.chdir("..")

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report

from src.pipeline.sabia import PipelineSabiaBinary, PipelineSabiaMultilabel, SabiaConfig
from src.utils.path import DATA_PATH
from src.utils.report import multilabel_report

In [ ]:
API_KEY = os.environ.get("MARITACA_API_KEY")

## Load Data

In [ ]:
df = pd.read_csv(f"{DATA_PATH}/ited-br/processed/ited-br.csv").sample(1)

In [ ]:
LABEL_COLUMNS = ["insult", "obscene", "ideology", "lgbtqphobia", "racism", "sexism", "xenophobia"]

# Binary Classification

In [ ]:
BINARY_PROMPT = """
Você é um classificador de tweets ofensivos. Dado um tweet, responda apenas com 1 se for ofensivo ou com 0 se não for ofensivo. Aqui estão 2 exemplos de tweets e suas classificações (entre parênteses):
1. "Minas Gerais é o estado com maior número de denúncias de crime e assédio eleitoral. A maioria esmagadora das denúncias tem conexão com a campanha de Bolsonaro, com a anuência do governador, que também é acusado de fazer assédio a prefeitos e políticos mineiros." (0)
2. "@user @user Não adianta derrubar! Todo mundo já sabe: Lula é ladrão. homicida, corrupto, mentiroso, ateu, terrorista, chefe de organizações criminosas!" (1)
""".strip()

In [ ]:
config_bn = SabiaConfig(
    api_key=API_KEY,
    label_columns=["offensive"],
    prompt=BINARY_PROMPT,
    max_tokens=2
)

In [ ]:
pipeline_bn = PipelineSabiaBinary(config_bn)
y_pred = pipeline_bn.predict(df, "text")
y_true = df["offensive"]
print(classification_report(y_true, y_pred))

# Multilabel Classification (Batch API)

In [ ]:
MULTILABEL_PROMPT = """
Você é um classificador de tweets ofensivos. Dado um tweet, responda com os seguintes rótulos (0 para ausência, 1 para presença):
- insult: insulto, palavrões ou ofensas cujo objetivo é provocar, humilhar ou atacar alguém verbalmente.
- obscene: conteúdo vulgar, sexualmente explícito ou pornográfico.
- ideology: discurso de ódio baseado em ideologias.
- lgbtqphobia: discurso de ódio contra identidade de gênero ou orientação sexual.
- racism: discurso de ódio baseado em raça, etnia ou cor.
- sexism: discurso de ódio baseado em gênero ou sexo.
- xenophobia: discurso de ódio contra estrangeiros ou pessoas de outras culturas ou nacionalidades.

Responda SOMENTE no formato:
insult:0,obscene:0,ideology:0,lgbtqphobia:0,racism:0,sexism:0,xenophobia:0
""".strip()

In [ ]:
config_ml = SabiaConfig(
    api_key=API_KEY,
    label_columns=LABEL_COLUMNS,
    prompt=MULTILABEL_PROMPT,
    max_tokens=100,
    workdir="./temp/batch_jobs/sabia"
)

In [ ]:
pipeline_ml = PipelineSabiaMultilabel(config_ml)
y_pred = pipeline_ml.predict(df, "text")
y_true = df[LABEL_COLUMNS].values
multilabel_report(y_true, y_pred, label_names=LABEL_COLUMNS)